# Lab 3 · NumPy và tư duy vector hoá

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 3**

> 💡 File → **Save a copy in Drive** trước khi sửa.

**Mục tiêu bài lab**

1. Tính dung lượng và vị trí phần tử từ `dtype`, `shape`, `strides`.
2. Chọn dữ liệu bằng chỉ mục, lát cắt và mask; phân biệt view với bản sao.
3. Viết phép tính theo hàng, cột bằng broadcasting và phát hiện phép tính sai dù code vẫn chạy.
4. Đo thời gian, diễn giải thống kê và lấy mẫu để kiểm tra dữ liệu.

Dữ liệu được tự tạo để có thể tính tay. Các ví dụ giá chỗ ở là **giả lập**, đơn vị **USD/đêm**, không phải kết quả phân tích Inside Airbnb.

## Cách làm và nộp bài

- Lab ở chế độ **✅ mở**: được dùng AI, cần đọc hiểu, kiểm chứng và khai báo ở cuối notebook.
- Trước khi chạy, ghi dự đoán vào ô trả lời. Thay `...` trong ô TODO bằng code của bạn.
- Chạy ô kiểm tra ngay sau mỗi bài. `assert` chỉ kiểm tra một số yêu cầu; phần giải thích vẫn cần được đọc và đánh giá.
- Hoàn thành các ô trả lời, chạy **Restart & Run all**, lưu notebook rồi nộp qua GitHub Classroom theo hướng dẫn của trợ giảng.

Notebook chưa điền TODO sẽ dừng ở ô kiểm tra; đây là trạng thái bình thường của đề bài.

## Dữ liệu dùng trong lab

`A` là mảng 4 hàng × 3 cột. Các bài có sửa dữ liệu sẽ làm trên bản sao của `A`.

In [40]:
import numpy as np
from timeit import repeat

A = np.array([[10, 12, 11],
              [20, 21, 24],
              [30, 33, 31],
              [40, 44, 42]], dtype=np.int64)
print(A)


[[10 12 11]
 [20 21 24]
 [30 33 31]
 [40 44 42]]


## Bài 1 · Đọc cấu trúc mảng

1. Ghi `shape`, `itemsize`, `nbytes` và `strides` của `A` trước khi chạy code.
2. `A[3, 1]` cách phần tử đầu bao nhiêu byte?
3. Nếu đổi sang `int32`, dung lượng dữ liệu và strides thay đổi thế nào?

`itemsize`: số byte mỗi phần tử. `nbytes`: dung lượng dữ liệu của mảng, chưa gồm thông tin quản lý đối tượng.

**Dự đoán và giải thích:**

- A: shape=(4, 3), itemsize=8), nbytes=4x3x8=96, strides=(24,8)
- Vị trí A[3, 1]: 3x24+1x8=80
- Khi đổi sang int32: itemsize giảm xuống 4 bytes, nbytes giảm còn 48 bytes, strides thay đổi thành (12, 4).

In [41]:
shape_a = A.shape
byte_moi_so = A.itemsize
byte_du_lieu = A.nbytes
buoc = A.strides
offset = 3 * buoc[0] + 1 * buoc[1]
A32 = A.astype(np.int32)


In [42]:
assert shape_a == (4, 3)
assert byte_moi_so == 8 and byte_du_lieu == 96
assert buoc == (24, 8) and offset == 80
assert A32.dtype == np.dtype("int32")
assert A32.nbytes == 48 and A32.strides == (12, 4)
np.testing.assert_array_equal(A32, A)
print("Bài 1: các kết quả kiểm tra đúng.")


Bài 1: các kết quả kiểm tra đúng.


## Bài 2 · Lát cắt và view

Tạo `V` gồm hàng 1, 3 và cột 0, 2 của `A`, **bằng một lát cắt cho mỗi chiều**.

1. Dự đoán giá trị, shape và strides của `V`.
2. `V[1, 1]` trỏ tới ô nào trong `A`? Ô đó cách đầu dữ liệu của `A` bao nhiêu byte?
3. Kiểm tra `V` có dùng chung dữ liệu với `A` không.

Dùng `np.shares_memory(A, V)` để kiểm tra việc dùng chung bộ nhớ.

**Dự đoán:** giá trị [[20, 24], [40, 42]]; shape (2, 2); strides (48, 16); ô trong A V[1, 1]; độ lệch 3x24+2x8=88.

In [43]:
# TODO — chưa sửa giá trị trong V
V = A[1::2, ::2]
shape_v = V.shape
strides_v = V.strides
offset_v11 = 3 * A.strides[0] + 2 * A.strides[1]
dung_chung = np.shares_memory(A, V)

In [44]:
np.testing.assert_array_equal(V, [[20, 24], [40, 42]])
assert shape_v == (2, 2) and strides_v == (48, 16)
assert offset_v11 == 88 and dung_chung
print("Bài 2: các kết quả kiểm tra đúng.")


Bài 2: các kết quả kiểm tra đúng.


## Bài 3 · View và bản sao

Đọc code dưới và dự đoán `B[0]` sau **mỗi** lệnh gán. Sau đó chạy để kiểm tra.

**Dự đoán:**

- Sau `C[0, 0] = -9`: B[0] = [10, 12, 11]
- Sau `V[0, 0] = -1`: B[0] = [10, -1, 11]
- Giải thích sự khác biệt: V được tạo ra bằng phương pháp Slicing tạo ra một view nên thay đổi ở V làm đổi B. C được tạo bằng Fancy Indexing, trả về một copy nên thay đổi trên C không ảnh hưởng tới B.

In [45]:
B = A.copy()
V = B[:, 1:3]
C = B[:, [1, 2]]
C[0, 0] = -9
print("Sau khi sửa C:", B[0])
V[0, 0] = -1
print("Sau khi sửa V:", B[0])


Sau khi sửa C: [10 12 11]
Sau khi sửa V: [10 -1 11]


Tạo `D` chứa cột 1 và 2 của `B` sao cho sửa `D` không làm đổi `B`.

In [46]:
# TODO
D = B[:, 1:3].copy()


In [47]:
np.testing.assert_array_equal(D, B[:, 1:3])
assert not np.shares_memory(D, B)
truoc = B.copy()
D[0, 0] = 999
np.testing.assert_array_equal(B, truoc)
np.testing.assert_array_equal(A[0], [10, 12, 11])
print("Bài 3: sửa D không làm đổi B.")


Bài 3: sửa D không làm đổi B.


## Bài 4 · Chọn phần tử

Không dùng vòng `for`:

1. Lấy các số chẵn trong `A` theo thứ tự từng hàng.
2. Lấy hai ô `A[0, 1]` và `A[2, 2]` bằng hai mảng chỉ mục.
3. Lấy cột 1, 2 ở mỗi hàng 0, 2 bằng `np.ix_`.

Ghi shape của ba kết quả và giải thích vì sao chúng khác nhau.

In [48]:
# TODO
so_chan = A[A % 2 == 0]
hai_o = A[[0, 2], [1, 2]]
bon_o = A[np.ix_([0, 2], [1, 2])]


In [49]:
np.testing.assert_array_equal(so_chan, [10, 12, 20, 24, 30, 40, 44, 42])
np.testing.assert_array_equal(hai_o, [12, 31])
np.testing.assert_array_equal(bon_o, [[12, 11], [33, 31]])
for ten, mang in [("so_chan", so_chan), ("hai_o", hai_o), ("bon_o", bon_o)]:
    assert not np.shares_memory(A, mang)
    print(ten, mang.shape)


so_chan (8,)
hai_o (2,)
bon_o (2, 2)


**Giải thích shape:** so_chan có shape (8,) vì lọc bằng Boolean Mask sẽ duỗi mảng thành 1 chiều. hai_o có shape (2,) vì ghép các chỉ mục tương ứng thành 2 vị trí điểm cụ thể: (0, 1) và (2, 2). bon_o có shape (2, 2) vì np.ix_ tạo ra lưới tọa độ hình chữ nhật từ tập các hàng [0, 2] và cột [1, 2].

## Bài 5 · Broadcasting

Tạo `K` qua hai bước:

1. Từ `A`, cộng 1, 2, 3 vào lần lượt các cột 0, 1, 2.
2. Trên kết quả vừa tính, cộng 10, 20, 30, 40 vào lần lượt các hàng 0, 1, 2, 3.

Ví dụ: `K[1, 2] = 24 + 3 + 20 = 47`.

Viết bằng NumPy, không dùng `for`. Ghi shape của từng mảng dùng để cộng.

In [50]:
b = np.array([1, 2, 3])
d = np.array([10, 20, 30, 40])
# TODO
K = A + b + d[:, np.newaxis]


In [51]:
np.testing.assert_array_equal(K, [[21, 24, 24], [41, 43, 47],
                                  [61, 65, 64], [81, 86, 85]])
np.testing.assert_array_equal(A[1], [20, 21, 24])
print("Bài 5: kết quả đúng và A không đổi.")


Bài 5: kết quả đúng và A không đổi.


**Giải thích:** shape của b (3,); của d (4,); shape dùng khi cộng theo hàng (4,1); vì sao

*   List item
*   List item

`A + d` lỗi Vì A có shape (4, 3) và d có shape (4,). Theo quy tắc broadcasting, NumPy so sánh kích thước từ phải sang trái (3 và 4) do không bằng nhau và không có chiều nào bằng 1 nên gây ra lỗi ValueError.

## Bài 6 · Sửa phép tính theo hàng

Mỗi hàng dưới đây là giá của một chỗ ở qua 3 kỳ thu thập (**giả lập, USD/đêm**).
Mỗi giá cần trừ đi trung bình của chính hàng đó.

Ví dụ: `[40, 50, 60]` → `[-10, 0, 10]`.

Code chạy được nhưng tính sai. Hãy sửa và giải thích lỗi.

In [52]:
gia = np.array([[40, 50, 60],
                [60, 60, 60],
                [30, 60, 90]], dtype=np.float64)
tb_sai = gia.mean(axis=1)
sai = gia - tb_sai
print(sai)


[[-10. -10.   0.]
 [ 10.   0.   0.]
 [-20.   0.  30.]]


In [53]:
# TODO
trung_binh_hang = gia.mean(axis=1, keepdims=True)
chenh_lech = gia - trung_binh_hang


In [54]:
assert trung_binh_hang.shape == (3, 1)
np.testing.assert_allclose(chenh_lech, [[-10, 0, 10], [0, 0, 0], [-30, 0, 30]])
np.testing.assert_allclose(chenh_lech.mean(axis=1), 0, atol=1e-12)
print("Bài 6: kết quả đúng.")


Bài 6: kết quả đúng.


**Giải thích:** `tb_sai` có shape (3,) nên khi trừ trực tiếp với gia có shape (3, 3), NumPy tự động broadcasting mở rộng tb_sai thành shape (1, 3) trừ theo cột.; cách sửa là sử dụng keepdims=True khi tính mean để giữ nguyên shape là (3, 1).

## Bài 7 · Đo thời gian

Ba cách dưới tính cùng phép nhân 2. Chạy phép đo với mảng nhỏ và mảng lớn, rồi trả lời:

1. Cách nào vẫn lặp trong Python?
2. Chuyển list sang ndarray có đủ để làm vòng `for` nhanh hơn không?
3. Kết quả đo có thay đổi theo số phần tử không? Vì sao?

Dữ liệu được tạo trước khi đo. Ba cách đều tạo kết quả mới; phép đo dùng thời gian nhỏ nhất trong 3 lần để giảm ảnh hưởng của nhiễu. Không yêu cầu một tỷ lệ nhanh/chậm cố định.

In [55]:
def nhan_list(xs):
    return [v * 2 for v in xs]

def nhan_for_array(x):
    return [v * 2 for v in x]

def nhan_array(x):
    return x * 2

for n in (10, 100_000):
    x = np.arange(n, dtype=np.int64)
    xs = x.tolist()
    np.testing.assert_array_equal(nhan_list(xs), nhan_array(x))
    np.testing.assert_array_equal(nhan_for_array(x), nhan_array(x))
    so_lan = 100 if n == 10 else 5
    print(f"\nn = {n:,}")
    for ten, ham, dau_vao in [("for trên list", nhan_list, xs),
                             ("for trên ndarray", nhan_for_array, x),
                             ("phép toán mảng", nhan_array, x)]:
        t = min(repeat(lambda: ham(dau_vao), number=so_lan, repeat=3)) / so_lan
        print(f"{ten}: {t * 1e6:.2f} µs/lần")



n = 10
for trên list: 0.46 µs/lần
for trên ndarray: 2.35 µs/lần
phép toán mảng: 1.31 µs/lần

n = 100,000
for trên list: 4211.85 µs/lần
for trên ndarray: 13002.91 µs/lần
phép toán mảng: 68.22 µs/lần


**Nhận xét từ phép đo của bạn:** Phương pháp sử dụng phép toán mảng nhanh hơn gấp nhiều lần so với vòng lặp for thông thường, sự chênh lệch này càng thể hiện rõ rệt khi số lượng phần tử lớn,

**Giải thích bằng cách thực thi và cách lưu dữ liệu:** Khi n lớn, phép toán mảng thực hiện tính toán song song với bộ nhớ liền kề, loại bỏ hoàn toàn chi phí kiểm tra kiểu động của Python nên thời gian xử lý nhanh hơn nhiều.

## Bài 8 · Thống kê giá và lấy mẫu

Giá của 5 chỗ ở (**giả lập, USD/đêm**): `[40, 50, 60, 70, 380]`.

1. Tính trung bình, trung vị, độ lệch chuẩn bằng NumPy.
2. Tính tỷ lệ chỗ ở có giá không quá 70 USD.
3. Lấy ngẫu nhiên 3 dòng, không lặp lại dòng, với seed 42. Tính trung bình mẫu.

Dùng `rng.choice(len(gia_dem), size=3, replace=False)` để lấy chỉ mục. `replace=False` không chọn trùng một dòng.

In [56]:
gia_dem = np.array([40, 50, 60, 70, 380], dtype=np.float64)
# TODO
trung_binh = np.mean(gia_dem)
trung_vi = np.median(gia_dem)
do_lech_chuan = np.std(gia_dem)
ty_le = np.mean(gia_dem <= 70)
rng = np.random.default_rng(42)
chi_muc = rng.choice(len(gia_dem), size=3, replace=False)
mau = gia_dem[chi_muc]
tb_mau = np.mean(mau)


In [57]:
assert trung_binh == 120 and trung_vi == 60
np.testing.assert_allclose(do_lech_chuan, np.sqrt(17000))
assert ty_le == 0.8
assert chi_muc.shape == (3,) and np.unique(chi_muc).size == 3
assert np.issubdtype(chi_muc.dtype, np.integer)
assert np.all((chi_muc >= 0) & (chi_muc < gia_dem.size))
np.testing.assert_array_equal(mau, gia_dem[chi_muc])
assert np.isclose(tb_mau, sum(mau) / 3)
print("Giá trong mẫu:", mau, "| Trung bình mẫu:", tb_mau)


Giá trong mẫu: [380.  40.  70.] | Trung bình mẫu: 163.33333333333334


**Trả lời ngắn:**

- Trong báo cáo bài tập lớn, bạn dùng trung bình hay trung vị để mô tả mức giá điển hình của dãy này? Vì sao? dùng trung vị vì mảng chứa giá trị ngoại lệ quá lớn làm kéo lệch đại lượng trung bình
- Có nên xoá giá 380 chỉ vì nó cao hơn các giá còn lại không? Không nên nếu đây là dữ liệu chính xác
- Trung bình mẫu có bằng trung bình cả 5 giá không? Seed cố định có bảo đảm mẫu đại diện không? Trung bình mẫu không bằng trung bình toàn bộ. Cố định Seed giúp tái lập kết quả ngẫu nhiên.
- Khi đã có toàn bộ dữ liệu hợp lệ, lấy mẫu có cần thiết để tính giá trung bình không? Khi nào việc đọc một mẫu dòng vẫn hữu ích? Không cần thiết lấy mẫu để tính giá trung bình nếu đã có dữ liệu đầy đủ. Tuy nhiên, việc lấy ra một vài dòng mẫu vẫn hữu ích khi cần kiểm tra nhanh cấu trúc dữ liệu, định dạng hoặc chạy thử thuật toán để tiết kiệm thời gian tính toán.

## Khai báo sử dụng AI

- **Công cụ đã dùng:** Không có
- **Bài đã dùng AI và yêu cầu chính:**
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:**

Không chỉ ghi “code chạy được”; nêu phép tính tay, shape hoặc ví dụ đối chiếu đã dùng.

---

## Tóm tắt bài lab

| Nội dung chính | Cần tự giải thích được |
|---|---|
| Shape, dtype, strides | Dung lượng dữ liệu và vị trí một phần tử |
| Lát cắt, view và bản sao | Sửa kết quả có làm đổi mảng gốc không |
| Mask và mảng chỉ mục | Những ô nào được chọn, shape kết quả |
| Broadcasting và axis | Mỗi phần tử được tính với giá trị nào |
| Đo thời gian | Vì sao phép toán trên mảng khác vòng for Python |
| Thống kê và lấy mẫu | Con số có phù hợp với câu hỏi trong bài tập lớn không |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

**Bài sau:** pandas — làm việc với bảng có tên cột và nhiều kiểu dữ liệu.